# Projeto Integrador IV — Entregável 02
## Dataset e Modelagem de Dados para IA

**Projeto:** Gestão Preditiva de Estoque com Inteligência Artificial  
**Integrantes:** Carlos Eduardo, Adryan Kevin, Eduardo Mendonça, Gabriel e Kevin Maykel


## 1. Contextualização

O projeto pretende utilizar dados históricos de estoque para, futuramente, apoiar previsões de demanda. Essas informações também poderão ajudar o responsável a identificar antecipadamente possíveis riscos de falta ou excesso de produtos.

Nesta etapa, o objetivo é apenas definir quais dados serão necessários para formar a base do projeto. Não será desenvolvido nenhum modelo neste entregável.


## 2. Definição do dataset

O dataset representará informações históricas dos produtos e de suas movimentações de saída. Cada registro mostrará a situação de um produto em uma data de referência.

Como ainda não temos acesso aos dados reais de uma empresa, será utilizada uma **pequena amostra simulada para fins acadêmicos**. Os valores foram definidos manualmente de forma coerente apenas para demonstrar a estrutura do dataset.


## 3. Features escolhidas

Foram escolhidas exatamente cinco features principais:

| Feature | Descrição | Justificativa |
|---|---|---|
| `estoque_atual` | Quantidade disponível do produto na data de referência. | Permite comparar a disponibilidade com o consumo e apoiar decisões de reposição. |
| `saidas_7_dias` | Quantidade retirada ou vendida nos últimos 7 dias. | Representa o comportamento mais recente da demanda. |
| `saidas_30_dias` | Quantidade retirada ou vendida nos últimos 30 dias. | Mostra uma tendência de consumo mais estável que a observação semanal. |
| `preco` | Preço atual de venda do produto, em reais. | Mudanças no preço podem influenciar a quantidade procurada. |
| `promocao` | Indica se o produto está em promoção: 1 para sim e 0 para não. | Promoções podem aumentar temporariamente as saídas do produto. |

As colunas `data_referencia` e `produto` serão usadas somente para identificar cada registro. Elas não fazem parte das cinco features principais definidas nesta entrega.


## 4. Pequena amostra do dataset

A amostra contém 12 registros simulados. Alguns produtos aparecem em duas datas para representar o acompanhamento histórico. O código abaixo serve somente para organizar e exibir a tabela no Google Colab.


In [ ]:
import pandas as pd

# Pequena amostra simulada para fins acadêmicos
dados = [
    ["2026-07-01", "Café 500g", 180, 63, 240, 18.90, 0],
    ["2026-07-08", "Café 500g", 145, 74, 259, 16.90, 1],
    ["2026-07-01", "Arroz 5kg", 240, 48, 190, 29.90, 0],
    ["2026-07-08", "Arroz 5kg", 210, 51, 198, 29.90, 0],
    ["2026-07-01", "Feijão 1kg", 190, 42, 166, 9.49, 0],
    ["2026-07-08", "Feijão 1kg", 160, 58, 188, 8.49, 1],
    ["2026-07-01", "Leite UHT 1L", 320, 96, 370, 5.79, 0],
    ["2026-07-08", "Leite UHT 1L", 270, 110, 405, 5.19, 1],
    ["2026-07-01", "Detergente 500ml", 150, 28, 112, 3.39, 0],
    ["2026-07-08", "Detergente 500ml", 128, 31, 120, 3.39, 0],
    ["2026-07-01", "Refrigerante 2L", 200, 70, 265, 10.49, 0],
    ["2026-07-08", "Refrigerante 2L", 155, 92, 310, 8.99, 1],
]

colunas = [
    "data_referencia",
    "produto",
    "estoque_atual",
    "saidas_7_dias",
    "saidas_30_dias",
    "preco",
    "promocao",
]

df = pd.DataFrame(dados, columns=colunas)


: 

In [ ]:
# Exibição da amostra em formato de tabela
df


## 5. Conclusão

As cinco features definidas formam uma base inicial para registrar a situação do estoque e o histórico recente de saídas. Nas próximas etapas do projeto, esses dados poderão ser ampliados e utilizados para apoiar previsões e decisões relacionadas ao estoque.


# Projeto Integrador IV — Entregável 03
## Prova de Conceito — Machine Learning Clássico

O objetivo é estimar a **demanda dos próximos 7 dias** com base no comportamento recente do produto. Como o resultado é uma quantidade numérica, utilizaremos regressão linear.


## 1. Ampliação do dataset

A amostra anterior é pequena. Por isso, serão gerados 120 registros simulados, mantendo os mesmos produtos e features. A variável-alvo `demanda_futura` combina saídas recentes, histórico de 30 dias, preço, promoção, período e uma pequena variação aleatória.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

np.random.seed(42)

# Produto, saída semanal típica e preço normal
produtos_ml = [
    ["Café 500g", 68, 18.90],
    ["Arroz 5kg", 50, 29.90],
    ["Feijão 1kg", 48, 9.49],
    ["Leite UHT 1L", 100, 5.79],
    ["Detergente 500ml", 30, 3.39],
    ["Refrigerante 2L", 78, 10.49],
]

registros = []
datas = pd.date_range("2026-02-18", periods=20, freq="7D")

for produto, saida_base, preco_base in produtos_ml:
    for semana, data in enumerate(datas):
        promocao = int(np.random.random() < 0.20)
        periodo = 1 + 0.08 * np.sin(2 * np.pi * semana / 10)
        preco = preco_base * (0.90 if promocao else 1)
        preco = round(preco * np.random.normal(1, 0.01), 2)

        saidas_7 = saida_base * periodo * (1.15 if promocao else 1)
        saidas_7 = max(1, int(round(saidas_7 + np.random.normal(0, 4))))
        saidas_30 = max(saidas_7, int(round(saidas_7 * 4 + np.random.normal(0, 10))))
        estoque = max(0, int(round(saidas_7 * np.random.uniform(2, 6))))

        demanda = 0.55 * saidas_7 + 0.35 * (saidas_30 / 4)
        demanda += 0.10 * saida_base * promocao
        demanda += 0.15 * saida_base * ((preco_base - preco) / preco_base)
        demanda += np.random.normal(0, 5)

        registros.append([
            data, produto, estoque, saidas_7, saidas_30,
            preco, promocao, max(1, int(round(demanda)))
        ])

df_ml = pd.DataFrame(registros, columns=[
    "data_referencia", "produto", "estoque_atual", "saidas_7_dias",
    "saidas_30_dias", "preco", "promocao", "demanda_futura"
])

print("Registros gerados:", len(df_ml))
df_ml.head()


## 2. Features, alvo e separação dos dados

`X` contém as cinco features do Entregável 02 e `y` contém a demanda futura. Usaremos 80% dos registros para o modelo aprender e 20% para avaliar exemplos não usados no treinamento.


In [ ]:
features = [
    "estoque_atual", "saidas_7_dias", "saidas_30_dias",
    "preco", "promocao"
]

X = df_ml[features]
y = df_ml["demanda_futura"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Treino:", len(X_treino), "registros")
print("Teste:", len(X_teste), "registros")


## 3. Treinamento, previsões e avaliação

`fit` é o momento em que o modelo aprende com os dados de treino. `predict` gera as previsões. O MAE informa o erro médio em unidades: quanto menor, mais próximas estão as previsões dos valores reais.


In [ ]:
modelo = LinearRegression()
modelo.fit(X_treino, y_treino)
previsoes = modelo.predict(X_teste)

mae = mean_absolute_error(y_teste, previsoes)
comparacao = pd.DataFrame({
    "demanda_real": y_teste.values,
    "demanda_prevista": np.round(previsoes, 1),
})

print(f"MAE: {mae:.2f} unidades")
comparacao.head(10)


## 4. Análise simples de risco de estoque

A regra será: estoque menor que a previsão indica **possível falta**; estoque maior que 1,8 vez a previsão indica **possível excesso**; os demais casos são considerados **adequados**.


In [ ]:
risco = df_ml.loc[X_teste.index, ["produto", "estoque_atual"]].copy()
risco["demanda_prevista"] = np.round(previsoes, 1)

risco["situacao"] = np.select(
    [
        risco["estoque_atual"] < risco["demanda_prevista"],
        risco["estoque_atual"] > risco["demanda_prevista"] * 1.8,
    ],
    ["Possível falta", "Possível excesso"],
    default="Adequado",
)

risco.head(10)


## 5. Conclusão

Esta prova de conceito demonstra o fluxo básico: dataset → features e alvo → treino e teste → modelo → previsão → MAE → aplicação simples ao estoque. O resultado é acadêmico e utiliza dados simulados.


# Entregável 04 — Otimização com Computação Evolutiva

Nesta etapa, a previsão de demanda do Entregável 03 é usada para apoiar uma decisão de reposição. O objetivo é distribuir um orçamento limitado entre os produtos para reduzir o risco de falta de itens. Trata-se de uma **prova de conceito com dados simulados para fins acadêmicos**.

Fluxo do projeto: dados de estoque (Entregável 02) → previsão de demanda por Machine Learning (Entregável 03) → escolha de reposição por algoritmo genético (Entregável 04).

## 1. Objetivo da otimização

Cada solução representa as quantidades a comprar de cada produto. Procuramos uma combinação que respeite o orçamento e atenda o máximo possível da necessidade de reposição. Assim, quanto maior a parte da demanda prevista que for coberta e menor a quantidade de produtos ainda em risco, melhor será a solução.

## 2. Dados utilizados

São usados os produtos, estoques e preços simulados do Entregável 03. O modelo já treinado naquele entregável (`modelo`) gera uma previsão de demanda para cada produto. Para tornar a compra comparável, a simulação considera um ciclo de reposição de **quatro semanas**: a necessidade é o que falta para cobrir quatro vezes a demanda semanal prevista. Não é criado nem treinado um novo modelo de Machine Learning.

In [ ]:
# Último registro simulado de cada produto: cenário usado na otimização
cenario = (df_ml.sort_values("data_referencia")
           .groupby("produto", as_index=False)
           .tail(1)
           .copy())

# A previsão vem do modelo treinado no Entregável 03.
cenario["demanda_prevista_semanal"] = np.maximum(
    1, np.round(modelo.predict(cenario[features]))
).astype(int)

# Configuração acadêmica: deseja-se cobertura para um ciclo de quatro semanas.
SEMANAS_DO_CICLO = 4
cenario["demanda_do_ciclo"] = (
    cenario["demanda_prevista_semanal"] * SEMANAS_DO_CICLO
)
cenario["necessidade_reposicao"] = np.maximum(
    0, cenario["demanda_do_ciclo"] - cenario["estoque_atual"]
).astype(int)

# O preço é usado como preço unitário simulado de compra nesta prova de conceito.
cenario = cenario[[
    "produto", "estoque_atual", "preco", "demanda_prevista_semanal",
    "demanda_do_ciclo", "necessidade_reposicao"
]].sort_values("produto").reset_index(drop=True)

cenario


## 3. Definição do problema

O orçamento máximo desta simulação acadêmica é de **R$ 1.000,00**. Cada gene de uma solução indica a quantidade a comprar de um produto, na mesma ordem da tabela abaixo. A compra não pode ultrapassar a necessidade calculada nem o orçamento disponível.

In [ ]:
ORCAMENTO = 1_000.00  # Configuração da simulação acadêmica

produtos = cenario["produto"].to_numpy()
precos = cenario["preco"].to_numpy(dtype=float)
necessidades = cenario["necessidade_reposicao"].to_numpy(dtype=int)

print(f"Orçamento disponível: R$ {ORCAMENTO:,.2f}")
print(f"Custo para atender toda a necessidade: R$ {(necessidades * precos).sum():,.2f}")
pd.DataFrame({
    "Produto": produtos,
    "Preço unitário (R$)": precos,
    "Quantidade necessária": necessidades,
})


## 4. Solução ingênua

Como referência, será utilizada uma regra sem algoritmo genético: dividir igualmente o orçamento entre os produtos. Para cada item, compra-se o máximo de unidades que cabe na sua parcela, sem ultrapassar sua necessidade. Essa regra simples não considera que alguns produtos podem ter uma necessidade maior ou um preço diferente.

In [ ]:
def ajustar_ao_orcamento(solucao):
    """Limita quantidades à necessidade e remove unidades até caber no orçamento."""
    solucao = np.minimum(np.maximum(np.asarray(solucao, dtype=int), 0), necessidades)

    while (solucao * precos).sum() > ORCAMENTO:
        candidatos = np.where(solucao > 0)[0]
        if len(candidatos) == 0:
            break
        # Remove uma unidade do item mais caro para voltar ao orçamento.
        indice = candidatos[np.argmax(precos[candidatos])]
        solucao[indice] -= 1
    return solucao


def avaliar(solucao):
    """Retorna os indicadores e a pontuação de uma estratégia de reposição."""
    solucao = ajustar_ao_orcamento(solucao)
    custo = float((solucao * precos).sum())
    atendido = np.minimum(solucao, necessidades)
    total_necessario = int(necessidades.sum())
    total_atendido = int(atendido.sum())
    percentual_atendido = 100 * total_atendido / total_necessario if total_necessario else 100
    produtos_em_risco = int(np.sum(solucao < necessidades))

    # Fitness é a pontuação que permite ao algoritmo reconhecer boas combinações.
    # Mais demanda atendida e menos produtos em risco resultam em maior pontuação.
    fitness = percentual_atendido * 10 - produtos_em_risco * 5
    return {
        "solucao": solucao,
        "custo": custo,
        "atendido": total_atendido,
        "percentual_atendido": percentual_atendido,
        "produtos_em_risco": produtos_em_risco,
        "fitness": fitness,
    }


def solucao_ingenua():
    parcela_por_produto = ORCAMENTO / len(produtos)
    quantidades = np.floor(parcela_por_produto / precos).astype(int)
    return ajustar_ao_orcamento(quantidades)

resultado_ingenuo = avaliar(solucao_ingenua())
resultado_ingenuo


## 5. Algoritmo genético

O algoritmo começa com uma população de estratégias aleatórias. Em cada geração, ele avalia as estratégias pela função de **fitness**, mantém as melhores (seleção), combina pares delas (cruzamento) e altera ocasionalmente uma quantidade (mutação). As novas soluções continuam respeitando o orçamento por meio da função de ajuste.

In [ ]:
# Parâmetros simples do algoritmo genético
TAMANHO_POPULACAO = 60
GERACOES = 100
TAXA_MUTACAO = 0.20
np.random.seed(42)  # Permite reproduzir a demonstração


def criar_individuo():
    """Cria uma possível compra e a torna viável para o orçamento."""
    candidato = np.array([
        np.random.randint(0, necessidade + 1) if necessidade > 0 else 0
        for necessidade in necessidades
    ])
    return ajustar_ao_orcamento(candidato)


def cruzar(pai, mae):
    """Cria um filho escolhendo, para cada produto, um valor de um dos pais."""
    mascara = np.random.rand(len(produtos)) < 0.5
    filho = np.where(mascara, pai, mae)
    return ajustar_ao_orcamento(filho)


def mutar(individuo):
    """Altera a quantidade de compra de um produto de forma pequena."""
    filho = individuo.copy()
    if np.random.rand() < TAXA_MUTACAO:
        indice = np.random.randint(len(produtos))
        variacao = np.random.randint(-10, 11)
        filho[indice] = max(0, filho[indice] + variacao)
    return ajustar_ao_orcamento(filho)


populacao = [criar_individuo() for _ in range(TAMANHO_POPULACAO)]
melhor_por_geracao = []

for geracao in range(GERACOES):
    populacao = sorted(populacao, key=lambda item: avaliar(item)["fitness"], reverse=True)
    melhor_por_geracao.append(avaliar(populacao[0])["fitness"])

    # Seleção: os 20 melhores formam o grupo de pais.
    pais = populacao[:20]
    nova_populacao = pais[:2]  # Elitismo: preserva as duas melhores soluções.

    while len(nova_populacao) < TAMANHO_POPULACAO:
        pai, mae = pais[np.random.randint(len(pais))], pais[np.random.randint(len(pais))]
        filho = mutar(cruzar(pai, mae))
        nova_populacao.append(filho)
    populacao = nova_populacao

melhor_solucao = max(populacao, key=lambda item: avaliar(item)["fitness"])
resultado_otimizado = avaliar(melhor_solucao)

print(f"Melhor fitness encontrado: {resultado_otimizado['fitness']:.2f}")
print(f"Fitness inicial: {melhor_por_geracao[0]:.2f}")
print(f"Fitness final: {melhor_por_geracao[-1]:.2f}")


## 6. Melhor solução encontrada

A tabela apresenta a recomendação de compra encontrada pelo algoritmo. `Saldo de risco` é a quantidade que ainda não pôde ser coberta no ciclo por causa do orçamento limitado.

In [ ]:
melhor_tabela = cenario[["produto", "necessidade_reposicao"]].copy()
melhor_tabela["quantidade_recomendada"] = resultado_otimizado["solucao"]
melhor_tabela["saldo_de_risco"] = (
    melhor_tabela["necessidade_reposicao"] - melhor_tabela["quantidade_recomendada"]
)
melhor_tabela["custo_por_produto"] = (
    melhor_tabela["quantidade_recomendada"] * precos
).round(2)

print(f"Custo total: R$ {resultado_otimizado['custo']:,.2f}")
print(f"Orçamento disponível: R$ {ORCAMENTO:,.2f}")
print(f"Orçamento utilizado: {resultado_otimizado['custo'] / ORCAMENTO * 100:.1f}%")
print(f"Demanda atendida: {resultado_otimizado['percentual_atendido']:.1f}%")
print(f"Produtos que permanecem em risco: {resultado_otimizado['produtos_em_risco']}")
print(f"Fitness: {resultado_otimizado['fitness']:.2f}")
melhor_tabela


## 7. Comparação

Os indicadores abaixo são produzidos pelo código e permitem comparar a regra fixa com a estratégia encontrada pelo algoritmo genético.

In [ ]:
import pandas as pd

comparacao_otimizacao = pd.DataFrame({
    "Indicador": [
        "Custo total (R$)",
        "Demanda atendida (unidades)",
        "Demanda atendida (%)",
        "Produtos em risco",
        "Fitness",
    ],
    "Solução ingênua": [
        round(resultado_ingenuo["custo"], 2),
        resultado_ingenuo["atendido"],
        round(resultado_ingenuo["percentual_atendido"], 1),
        resultado_ingenuo["produtos_em_risco"],
        round(resultado_ingenuo["fitness"], 2),
    ],
    "Solução otimizada": [
        round(resultado_otimizado["custo"], 2),
        resultado_otimizado["atendido"],
        round(resultado_otimizado["percentual_atendido"], 1),
        resultado_otimizado["produtos_em_risco"],
        round(resultado_otimizado["fitness"], 2),
    ],
})
comparacao_otimizacao


## 8. Conclusão

A solução otimizada é comparada automaticamente à solução ingênua na tabela anterior. O algoritmo genético explora diferentes distribuições do mesmo orçamento e favorece as que atendem mais unidades previstas e deixam menos produtos em risco. Como os dados, o orçamento e o ciclo de quatro semanas são simulados, esta é uma demonstração acadêmica do uso de Computação Evolutiva na gestão preditiva de estoque.

# Entregável 05 — Integração e Pipeline de Dados

Nesta etapa final, consolidamos os aprendizados anteriores (Machine Learning e Computação Evolutiva) em um único fluxo contínuo. Este pipeline recebe os dados brutos de estoque de um lado e devolve uma ordem de compra otimizada do outro, simulando um ambiente real de tomada de decisão.

## 1. Modularização do Fluxo (Refatoração)

Vamos encapsular as lógicas desenvolvidas nas etapas 3 e 4 em funções reutilizáveis. Isso limpa o código e permite que o sistema processe novos cenários (ex: atualização semanal do estoque) com uma única chamada de função.

In [5]:
def treinar_modelo_previsao(df_historico, features):
    """Treina e retorna o modelo de regressão linear para previsão de demanda."""
    X = df_historico[features]
    y = df_historico["demanda_futura"]
    modelo = LinearRegression()
    modelo.fit(X, y)
    return modelo

def prever_necessidades(modelo, df_atual, features, semanas_ciclo=4):
    """Calcula a necessidade de reposição baseada nas previsões."""
    df_result = df_atual.copy()
    
    # Prever demanda semanal
    demanda_semanal = np.maximum(1, np.round(modelo.predict(df_atual[features]))).astype(int)
    
    # Projetar para o ciclo e calcular o que falta
    demanda_ciclo = demanda_semanal * semanas_ciclo
    necessidade = np.maximum(0, demanda_ciclo - df_atual["estoque_atual"]).astype(int)
    
    df_result["demanda_prevista_semanal"] = demanda_semanal
    df_result["necessidade_reposicao"] = necessidade
    return df_result

def otimizar_compras(produtos, precos, necessidades, orcamento_disp, tamanho_populacao=60, geracoes=100, taxa_mutacao=0.2):
    """Utiliza algoritmo genético para decidir as compras otimizadas sob restrição de orçamento."""
    
    def ajustar(solucao):
        solucao = np.minimum(np.maximum(np.asarray(solucao, dtype=int), 0), necessidades)
        while (solucao * precos).sum() > orcamento_disp:
            candidatos = np.where(solucao > 0)[0]
            if len(candidatos) == 0: break
            indice = candidatos[np.argmax(precos[candidatos])]
            solucao[indice] -= 1
        return solucao
        
    def fitness(solucao):
        solucao = ajustar(solucao)
        atendido = np.minimum(solucao, necessidades).sum()
        total_necessario = necessidades.sum()
        pct = 100 * atendido / total_necessario if total_necessario else 100
        risco = np.sum(solucao < necessidades)
        return pct * 10 - risco * 5
        
    populacao = [ajustar([np.random.randint(0, n + 1) if n > 0 else 0 for n in necessidades]) for _ in range(tamanho_populacao)]
    
    for _ in range(geracoes):
        populacao.sort(key=fitness, reverse=True)
        pais = populacao[:20]
        nova_pop = pais[:2] # Elitismo
        while len(nova_pop) < tamanho_populacao:
            pai, mae = pais[np.random.randint(len(pais))], pais[np.random.randint(len(pais))]
            mascara = np.random.rand(len(produtos)) < 0.5
            filho = np.where(mascara, pai, mae)
            if np.random.rand() < taxa_mutacao:
                idx = np.random.randint(len(produtos))
                filho[idx] = max(0, filho[idx] + np.random.randint(-10, 11))
            nova_pop.append(ajustar(filho))
        populacao = nova_pop
        
    melhor = max(populacao, key=fitness)
    return ajustar(melhor)

## 2. Executando o Pipeline de Ponta a Ponta

Agora vamos simular um 'Fechamento de Mês'. O sistema lerá os dados atuais, processará o Machine Learning e rodará a otimização evolutiva de uma vez só, entregando a sugestão de compras.

In [3]:
# 1. Definir o cenário (Orçamento e Features)
ORCAMENTO_COMPRAS = 1200.00
features_modelo = ["estoque_atual", "saidas_7_dias", "saidas_30_dias", "preco", "promocao"]

# 2. Treinar o modelo de ML com o histórico completo
modelo_producao = treinar_modelo_previsao(df_ml, features_modelo)

# 3. Pegar a "fotografia" atual do estoque (os dados mais recentes do nosso dataset)
estoque_hoje = df_ml.sort_values("data_referencia").groupby("produto", as_index=False).tail(1).copy().reset_index(drop=True)

# 4. Prever necessidades usando ML
necessidades_hoje = prever_necessidades(modelo_producao, estoque_hoje, features_modelo)

# 5. Otimizar as compras via Computação Evolutiva
array_produtos = necessidades_hoje["produto"].to_numpy()
array_precos = necessidades_hoje["preco"].to_numpy(dtype=float)
array_necessidades = necessidades_hoje["necessidade_reposicao"].to_numpy(dtype=int)

compras_sugeridas = otimizar_compras(array_produtos, array_precos, array_necessidades, orcamento_disp=ORCAMENTO_COMPRAS)

# 6. Compilar Resultado
relatorio = necessidades_hoje[["produto", "estoque_atual", "necessidade_reposicao"]].copy()
relatorio["compra_recomendada"] = compras_sugeridas
relatorio["saldo_risco_futuro"] = relatorio["necessidade_reposicao"] - compras_sugeridas
relatorio["investimento_necessario"] = compras_sugeridas * array_precos

print("Pipeline Concluído com Sucesso! ✓\n")
print(f"Orçamento Disponibilizado: R$ {ORCAMENTO_COMPRAS:,.2f}")
print(f"Investimento Utilizado: R$ {relatorio['investimento_necessario'].sum():,.2f}\n")
relatorio

NameError: name 'df_ml' is not defined

## 3. Visualização de Impacto

Gráfico que ilustra o resultado final da nossa solução: onde o orçamento foi aplicado e quais áreas de risco sobraram. A visualização ajuda gestores a decidirem se o orçamento precisa de reforço.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(relatorio["produto"]))
width = 0.35

ax.bar(x - width/2, relatorio["compra_recomendada"], width, label='Compra Recomendada (Atendida)', color='#4CAF50')
ax.bar(x + width/2, relatorio["saldo_risco_futuro"], width, label='Saldo em Risco (Sem Orçamento)', color='#F44336')

ax.set_ylabel('Unidades')
ax.set_title('Recomendação de Compra vs. Risco Residual por Produto')
ax.set_xticks(x)
ax.set_xticklabels(relatorio["produto"], rotation=15)
ax.legend()

plt.tight_layout()
plt.show()